In [1]:
from xml.etree import ElementTree as ET

In [ ]:
ttml_data ="""<?xml version="1.0" encoding="UTF-8"?>
<tt xmlns="http://www.w3.org/ns/ttml" xmlns:itunes="http://music.apple.com/lyrics">
  <body>
    <div itunes:song-part="Verse">
      <p begin="00:00:15.053" end="00:00:20.562">
        <span begin="00:00:15.053" end="00:00:15.522">I </span>
        <span begin="00:00:15.522" end="00:00:16.021">know </span>
        <span begin="00:00:16.021" end="00:00:16.437">that </span>
        <span begin="00:00:16.437" end="00:00:16.704">the </span>
        <span begin="00:00:16.704" end="00:00:17.104">bar </span>
        <span begin="00:00:17.104" end="00:00:17.789">closes </span>
        <span begin="00:00:17.789" end="00:00:18.256">at </span>
        <span begin="00:00:18.256" end="00:00:20.562">11</span>
      </p>
      <p begin="00:00:22.204" end="00:00:27.959">
        <span begin="00:00:22.204" end="00:00:22.490">But </span>
        <span begin="00:00:22.490" end="00:00:22.908">I </span>
        <span begin="00:00:22.908" end="00:00:23.340">hope </span>
        <span begin="00:00:23.340" end="00:00:23.639">you </span>
        <span begin="00:00:23.639" end="00:00:24.402">never </span>
        <span begin="00:00:24.402" end="00:00:25.386">finish </span>
        <span begin="00:00:25.386" end="00:00:25.884">that </span>
        <span begin="00:00:25.884" end="00:00:27.959">beer</span>
      </p>
      <p begin="00:03:37.809" end="00:03:42.469">
        <span begin="00:03:37.809" end="00:03:38.327">Kiss </span>
        <span begin="00:03:38.327" end="00:03:38.777">me </span>
        <span begin="00:03:38.777" end="00:03:39.228">and </span>
        <span begin="00:03:39.228" end="00:03:39.644">I </span>
        <span begin="00:03:39.644" end="00:03:40.161">might </span>
        <span begin="00:03:40.161" end="00:03:40.676">drop </span>
        <span begin="00:03:40.676" end="00:03:42.469">dead</span>
      </p>
    </div>
  </body>
</tt>
"""

In [5]:
root = ET.fromstring(ttml_data)

In [33]:
from datetime import datetime
def parse_time(time_str):
    # Parse the time string into a datetime object
    time_obj = datetime.strptime(time_str, "%H:%M:%S.%f")
    # Convert to total seconds
    total_seconds = time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second + time_obj.microsecond / 1e6
    return total_seconds

parse_time("00:00:15.053")

15.053

In [34]:
whisperx_json = []

for item in root.iter():
    if item.tag.endswith('p'):
        texts = []
        words = []
        p_attrib = item.attrib
        start = parse_time(p_attrib.get('begin'))
        end = parse_time(p_attrib.get('end'))
        for child in item:
            if child.tag.endswith('span'):
                texts.append(child.text.strip())
                words.append({
                    "word": child.text.strip(),
                    "start": parse_time(child.attrib.get('begin')),
                    "end": parse_time(child.attrib.get('end'))
                })
        whisperx_json.append({
            "text": ' '.join(texts),
            "start": start,
            "end": end,
            "words": words
        })

In [37]:
import json
json.dumps(whisperx_json, indent=4)
with open("output.json", "w") as f:
    json.dump(whisperx_json, f, indent=4)